# Fit DT Controller Parameters To Experimental Scan Results

This notebook follows the local workflow in `Clean up data to train DT.ipynb`, `Visualize the grid-scans.ipynb`, and `FD-prior cloud distribution calibration.ipynb`.

Workflow:
1. Load the measured Tap300 AlScN FD curves and extrapolate each near-contact branch to amplitude `A = 0`.
2. Build the FD-initialized DT scanner and fit PI controller parameters to the experimental grid scan traces.
3. Visualize the fitted PI values and representative experimental/DT trace alignment.
4. Fit pure data-driven trace models and DT residual alignment models, then export two publication-ready figures.


In [ ]:
from pathlib import Path
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from joblib import dump, load

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_ROOT = Path("/Users/richardmbp14/Library/CloudStorage/GoogleDrive-yu93liu@gmail.com/My Drive/Shared/Coding/Digital Twins - SPM")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = PROJECT_ROOT / "output" / "dt_controller_fit_figures"
CACHE_DIR = PROJECT_ROOT / "calibration_cache" / "dt_controller_fit"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FIT_PROFILE = "balanced"  # "preview", "balanced", or "paper"
REFIT_PHYSICS_GUIDED = False
REFIT_DATA_DRIVEN = False
RANDOM_SEED = 35

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 140,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# --- Gain-filter knob ----------------------------------------------
# Drop the highest I-gain values from the grid because they typically
# saturate the controller and produce flat, uninformative traces. With
# the AlScN dataset (igain_exp = [10, 30, 50, 100, 150]) we keep the
# first 3 = [10, 30, 50].
N_GAINS_KEEP = 3

# Cache tag combines fit profile and gain filter so caches do not
# silently reuse a stale (5-gain) fit.
CACHE_TAG = f"{FIT_PROFILE}_g{N_GAINS_KEEP}_rms"

PROFILE_CONFIG = {
    "preview": dict(n_train=24, n_test=12, n_grid_P=9, n_grid_I=9, n_lines_train=2, n_lines_test=4, maxiter_local=40, dd_max_conditions=360, n_estimators=250),
    "balanced": dict(n_train=80, n_test=30, n_grid_P=13, n_grid_I=13, n_lines_train=3, n_lines_test=6, maxiter_local=80, dd_max_conditions=900, n_estimators=450),
    "paper": dict(n_train=200, n_test=70, n_grid_P=21, n_grid_I=21, n_lines_train=4, n_lines_test=8, maxiter_local=180, dd_max_conditions=None, n_estimators=700),
}
CFG = PROFILE_CONFIG[FIT_PROFILE]
print("Project:", PROJECT_ROOT)
print("Figures:", FIG_DIR)
print("Cache:", CACHE_DIR)
print("Fit profile:", FIT_PROFILE, CFG)
print("Cache tag :", CACHE_TAG, " (N_GAINS_KEEP =", N_GAINS_KEEP, ")")


## 1. Load Experimental Grid Scans And FD Curves

The grid-scan file stores height, amplitude, and phase as trace/retrace channels. The drive conversion follows the earlier notebooks: voltage drive is converted to nm using the Igor topography header `AmpInvOLS`.


In [ ]:
import aespm as ae

fd_npz = np.load(PROJECT_ROOT / "output" / "Tap300_AlScN.npz")
grid_npz = np.load(PROJECT_ROOT / "data" / "260507-300kHz-AlScN.npz")
topo = ae.tools.load_ibw(PROJECT_ROOT / "data" / "Image0002.ibw")

fd_height = fd_npz["height"]
fd_amp = fd_npz["amp"]
fd_phase = fd_npz["phase"]
fd_drive_raw = fd_npz["drive"][:-1]
fd_drive_far_A = np.nanmean(fd_amp[:, -10:], axis=1)
fd_drive_nm = 0.8 * fd_drive_far_A

raw_grid = grid_npz["data"]
scan_rate = grid_npz["scan_rate"].astype(float)
drive_exp = grid_npz["drives"].astype(float) * (258.8 / 2.14) * topo.header["AmpInvOLS"] * 1e9
setpoint_exp = grid_npz["setpoints"].astype(float)
igain_exp = grid_npz["i_gain"].astype(float)

GRID_CHANNEL = 1
traces_height_raw = raw_grid[:, :, :, :, GRID_CHANNEL, 0:2, :]
traces_height = (traces_height_raw - np.nanmin(traces_height_raw, axis=-1, keepdims=True)) * 1e9
traces_A = raw_grid[:, :, :, :, GRID_CHANNEL, 2:4, :] * 1e9
traces_phi = raw_grid[:, :, :, :, GRID_CHANNEL, 4:6, :]

h_map = (topo.data[0] - np.nanmin(topo.data[0])) * 1e9

print("FD curves:", fd_height.shape, fd_amp.shape, fd_phase.shape)
print("FD drive range used for scanner (nm):", float(np.nanmin(fd_drive_nm)), float(np.nanmax(fd_drive_nm)))
print("Experimental traces:", traces_height.shape)
print("Experimental drive range (nm):", drive_exp)
print("Setpoints:", setpoint_exp)
print("I gains:", igain_exp)
print("Scan rates:", scan_rate)
print("Topo map:", h_map.shape)

# ---- Apply gain filter -------------------------------------------
# Keep only the lowest N_GAINS_KEEP I-gain values; drop the rest
# (the high-gain traces saturate and carry no information).
if N_GAINS_KEEP < traces_height.shape[3]:
    igain_exp_full = igain_exp.copy()
    igain_exp     = igain_exp[:N_GAINS_KEEP]
    traces_height = traces_height[:, :, :, :N_GAINS_KEEP]
    traces_A      = traces_A     [:, :, :, :N_GAINS_KEEP]
    traces_phi    = traces_phi   [:, :, :, :N_GAINS_KEEP]
    print(f"  -> Gain filter applied. Kept {N_GAINS_KEEP} of "
          f"{len(igain_exp_full)} I-gain values: {igain_exp}")
    print(f"     dropped: {igain_exp_full[N_GAINS_KEEP:]}")
    print(f"  -> New traces_height shape: {traces_height.shape}")
else:
    print("  -> No gain filter applied (N_GAINS_KEEP >= n_gain).")


## 1b.  Informative-trace checker

For each remaining `(speed, drive, setpoint, gain)` condition we run the
project's `classify_height_trace_pair` classifier on the experimental
trace/retrace pair and tally how many are informative. This both reports
the impact of the gain filter and stores an `informative_mask` that the
downstream fits will consume so we never waste local-PI fitting time on
flat traces.


In [ ]:
from codes.dt_trace_classifier import (
    TraceUsabilityConfig,
    classify_height_trace_pair,
)

usability_cfg = TraceUsabilityConfig(
    trim=10,
    min_mad=0.02,        # nm,    height MAD lower bound
    min_range=0.10,      # nm,    peak-to-trough range lower bound
    min_grad_mad=0.002,  # nm/px, per-pixel gradient MAD lower bound
    min_nonzero_frac=0.05,
)

n_speed, n_drive, n_setpoint, n_gain, _, n_pix = traces_height.shape
informative_mask = np.zeros((n_speed, n_drive, n_setpoint, n_gain), dtype=bool)
reason_counter = {}

for si in range(n_speed):
    for di in range(n_drive):
        for spi in range(n_setpoint):
            for gi in range(n_gain):
                info = classify_height_trace_pair(
                    traces_height[si, di, spi, gi, 0],
                    traces_height[si, di, spi, gi, 1],
                    cfg=usability_cfg,
                )
                cls = info.get("class", "unknown")
                informative_mask[si, di, spi, gi] = (cls == "informative")
                reason_counter[cls] = reason_counter.get(cls, 0) + 1

n_total = informative_mask.size
n_info  = int(informative_mask.sum())
print(f"informative conditions : {n_info} / {n_total}  "
      f"({100 * n_info / n_total:.1f}%)")
print(f"breakdown by class     : {reason_counter}")

# Per-gain breakdown so we can see if the filter actually removed the bad bins.
print()
for gi in range(n_gain):
    nf = int(informative_mask[:, :, :, gi].sum())
    nt = informative_mask[:, :, :, gi].size
    print(f"  Igain = {igain_exp[gi]:5.0f}  :  {nf:4d} / {nt:4d} informative "
          f"({100 * nf / nt:.1f}%)")


In [ ]:
# Quick visual sanity check: one informative + one uninformative trace.
fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.4), sharey=True)

info_idx = np.argwhere(informative_mask)
uninf_idx = np.argwhere(~informative_mask)
if len(info_idx) > 0 and len(uninf_idx) > 0:
    si, di, spi, gi = info_idx[len(info_idx) // 2]
    axes[0].plot(traces_height[si, di, spi, gi, 0], color="black",
                  lw=1.0, label="trace")
    axes[0].plot(traces_height[si, di, spi, gi, 1], color="0.55",
                  lw=1.0, label="retrace")
    axes[0].set_title(
        f"informative   "
        f"v={scan_rate[si]:g}, d={drive_exp[di]:.1f}, "
        f"sp={setpoint_exp[spi]:.2f}, Ig={igain_exp[gi]:.0f}",
        fontsize=8,
    )
    axes[0].set_xlabel("pixel"); axes[0].set_ylabel("height (nm)")
    axes[0].legend(frameon=False, fontsize=7)

    si, di, spi, gi = uninf_idx[len(uninf_idx) // 2]
    axes[1].plot(traces_height[si, di, spi, gi, 0], color="black", lw=1.0)
    axes[1].plot(traces_height[si, di, spi, gi, 1], color="0.55", lw=1.0)
    axes[1].set_title(
        f"NOT informative   "
        f"v={scan_rate[si]:g}, d={drive_exp[di]:.1f}, "
        f"sp={setpoint_exp[spi]:.2f}, Ig={igain_exp[gi]:.0f}",
        fontsize=8,
    )
    axes[1].set_xlabel("pixel")
fig.suptitle("Informative-trace checker: representative pair", fontsize=9)
fig.tight_layout()
fig


## 2. Build The FD-Prior DT Scanner

The measured FD curves are converted into a drive-dependent FD surface. `FDDriveSurface.from_measured_fd(..., extend_to_zero_amplitude=True)` performs the near-contact linear extrapolation to `A = 0`; the resulting surface is then wrapped in the existing `ScannerFD` controller.


In [ ]:
import importlib
import codes.Scanner_fixed_extended_fd as sfe
import codes.Scanner_numba_substeps_v3 as sn

importlib.reload(sfe)
importlib.reload(sn)
sn.install_numba_substep_methods(sfe.ScannerFD, verbose=True)

FDDriveSurface = sfe.FDDriveSurface
make_normalized_fd_lookup = sfe.make_normalized_fd_lookup
ScannerFD = sfe.ScannerFD

measured_fd = {
    float(fd_drive_nm[i]): {
        "d": fd_height[i],
        "A": fd_amp[i],
        "phi": fd_phase[i],
    }
    for i in range(len(fd_drive_nm))
}

surface = FDDriveSurface.from_measured_fd(
    measured_fd,
    x_mode="d_over_A0",
    common_range="union",
    n_x=1024,
    extend_to_zero_amplitude=True,
    extension_n_fit=5,
    extension_n_bridge=30,
    extension_phi_mode="linear",
    extension_F_mode="nearest",
)

params = {
    "k": 25,
    "A": float(np.nanmedian(fd_amp)),
    "d0": float(np.nanmax(fd_height) * 0.8),
    "R": 10,
    "H": 1e-19,
    "E_star": 1e9,
    "Q": 250,
}
conversions = {k: 1.0 for k in params}

scanner = ScannerFD(
    params=params,
    conversions=conversions,
    fd_model=make_normalized_fd_lookup(surface, conv_L=1.0, conv_A=1.0, conv_F=1.0),
)

print("A0 at experimental drive min/max:", scanner.get_A0_hat(drive_exp.min()), scanner.get_A0_hat(drive_exp.max()))
print("FD distance range at median drive:", scanner.distance_range(float(np.median(drive_exp))))
print("Example measure_from_fd:", scanner.measure_from_fd(5.0, drive_nm=float(np.median(drive_exp))))


In [ ]:
def save_figure(fig, stem):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png)
    fig.savefig(pdf)
    print("saved", png)
    print("saved", pdf)
    return png, pdf


def plot_fd_zero_extension(surface, measured_fd, drives_to_show=None):
    if drives_to_show is None:
        keys = np.array(sorted(measured_fd.keys()), dtype=float)
        target = np.linspace(drive_exp.min(), drive_exp.max(), 4)
        drives_to_show = [float(keys[np.argmin(np.abs(keys - t))]) for t in target]

    extended = surface.measured_fd_extended or measured_fd
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.15), constrained_layout=True)
    cmap = plt.get_cmap("viridis", len(drives_to_show))

    for j, drv in enumerate(drives_to_show):
        raw = measured_fd[drv]
        ext = extended[drv]
        color = cmap(j)
        axes[0].plot(raw["d"], raw["A"], color=color, lw=0.8, alpha=0.65)
        axes[0].plot(ext["d"], ext["A"], color=color, lw=1.2, label=f"{drv:.1f} nm")
        axes[1].plot(ext["d"], ext["phi"], color=color, lw=1.0)

    diag = surface.diagnostics()
    axes[2].plot(diag["drives"], diag["A0_by_drive"], "o-", ms=3, lw=1.0, color="black")
    axes[2].axvspan(drive_exp.min(), drive_exp.max(), color="tab:blue", alpha=0.10, lw=0)

    axes[0].axhline(0, color="0.2", lw=0.7, ls="--")
    axes[0].set_xlabel("distance d (nm)")
    axes[0].set_ylabel("amplitude A (nm)")
    axes[0].set_title("FD extension to A=0")
    axes[0].legend(frameon=False, loc="best", ncol=1)

    axes[1].set_xlabel("distance d (nm)")
    axes[1].set_ylabel("phase (deg)")
    axes[1].set_title("extended phase branch")

    axes[2].set_xlabel("drive (nm)")
    axes[2].set_ylabel("free amplitude A0 (nm)")
    axes[2].set_title("FD prior drive range")
    return fig

fig_fd_ext = plot_fd_zero_extension(surface, measured_fd)
save_figure(fig_fd_ext, "figure_s1_fd_zero_amplitude_extension")


## 3. Generate Scan Lines For Fitting

For PI fitting, the original notebook used measured map rows from `Image0002.ibw` as the scan-line truth. The code below keeps that option and also keeps synthetic line generation available for sensitivity checks.

**Improved scan-line builder (v2).**  The PI-fitting inputs are now drawn by *random sampling* from the topography rows (default 4× over-sampled relative to the per-iteration fit budget), linearly detrended and rebaselined to match the per-line baseline shift the experimental traces already received.  Synthetic lines remain available via `USE_SYNTHETIC_LINES = True` and now default to the `block` bootstrap mode which preserves local spatial correlation much better than the previous filtered-iid setting.


In [ ]:
# ---------------------------------------------------------------
# Improved scan-line builder.
# ---------------------------------------------------------------
# We use measured rows of the topography as PI-fitting inputs (instead of
# purely synthetic lines) because the DT-vs-experiment overlay shows the
# DT response is much closer to reality when the input height profile
# carries the same spatial-frequency content as the real scan.  Four
# improvements over the v1 setup:
#   - draw many more rows (default 4x the per-fit-iteration budget) so
#     local PI fits are averaged over a more representative roughness
#     sample;
#   - pick rows by random sampling (with a fixed seed) instead of
#     uniformly spaced row indices, which avoids structural artefacts of
#     the topo (terraces, defects) systematically landing on the same
#     rows every time;
#   - linearly detrend each row and rebaseline to pixel minimum, so the
#     input lines match the per-line baseline shift the experimental
#     traces already received;
#   - **NEW: rescale every input line so its detrended RMS matches the
#     median RMS of the experimental height traces.**  This guarantees
#     that the input profile carries the same magnitude of height
#     variation that the experiment actually delivered, not just the
#     same rate distribution.

USE_SYNTHETIC_LINES = False

# Sample size: many more rows than the inner fit needs, then the inner
# loop will average over a subset.  Override with FIT_LINE_OVERSAMPLE
# in the setup cell if you want a different multiplier.
FIT_LINE_OVERSAMPLE = 4
N_FIT_LINES = max(int(CFG["n_lines_test"]) * FIT_LINE_OVERSAMPLE, 16)
N_FIT_LINES = min(N_FIT_LINES, h_map.shape[0])    # cannot exceed n_rows


def _detrend(line):
    """Linear detrend + rebaseline to pixel minimum."""
    line = np.asarray(line, dtype=float).ravel()
    x = np.arange(line.size, dtype=float)
    finite = np.isfinite(line)
    if finite.sum() < 4:
        return line - np.nanmin(line)
    slope, intercept = np.polyfit(x[finite], line[finite], 1)
    detr = line - (slope * x + intercept)
    return detr - np.nanmin(detr)


def _exp_trace_rms(line):
    """RMS of a centered + detrended height trace in nm."""
    line = np.asarray(line, dtype=float).ravel()
    line = line[np.isfinite(line)]
    if line.size < 4:
        return np.nan
    x = np.arange(line.size, dtype=float)
    slope, intercept = np.polyfit(x, line, 1)
    centered = line - (slope * x + intercept)
    centered = centered - np.nanmean(centered)
    return float(np.sqrt(np.nanmean(centered ** 2)))


# ---- Target RMS from the experimental traces ----
# Compute RMS per (speed, drive, setpoint, gain, direction); restrict to
# the informative conditions so flat / saturated traces do not pull the
# target down.
_exp_rms_samples = []
for si in range(traces_height.shape[0]):
    for di in range(traces_height.shape[1]):
        for spi in range(traces_height.shape[2]):
            for gi in range(traces_height.shape[3]):
                if not informative_mask[si, di, spi, gi]:
                    continue
                for d in range(2):
                    r = _exp_trace_rms(traces_height[si, di, spi, gi, d])
                    if np.isfinite(r) and r > 0:
                        _exp_rms_samples.append(r)
_exp_rms_samples = np.array(_exp_rms_samples, dtype=float)
TARGET_RMS_NM = float(np.nanmedian(_exp_rms_samples)) if _exp_rms_samples.size else float("nan")
print(f"target RMS (median over informative exp. traces): {TARGET_RMS_NM:.3f} nm "
      f"  (5/95 pct: {np.nanpercentile(_exp_rms_samples, 5):.3f} / "
      f"{np.nanpercentile(_exp_rms_samples, 95):.3f} nm,  "
      f"n = {_exp_rms_samples.size})")


def _rms_match(line, target_rms=TARGET_RMS_NM, min_floor_frac=0.05):
    """Rescale a line so that its centered + detrended RMS equals
    target_rms.  Returns the rescaled line, rebaselined to min = 0."""
    detr = _detrend(line)
    centered = detr - np.nanmean(detr)
    rms = float(np.sqrt(np.nanmean(centered ** 2)))
    if not np.isfinite(rms) or rms < min_floor_frac * target_rms:
        # too flat or degenerate; return at floor magnitude
        return _detrend(line)
    scaled = centered * (target_rms / rms)
    return scaled - np.nanmin(scaled)


def _prepare_fit_line(line, target_rms=TARGET_RMS_NM):
    """Single entry point: detrend + RMS-match + rebaseline."""
    return _rms_match(line, target_rms=target_rms)


# ---- Build the fitting batch ----
rng = np.random.default_rng(RANDOM_SEED)
row_ids = np.sort(
    rng.choice(h_map.shape[0], size=int(N_FIT_LINES), replace=False)
)
h_line_exp = np.asarray(
    [_prepare_fit_line(h_map[r, :traces_height.shape[-1]]) for r in row_ids],
    dtype=float,
)

synthetic_lines = None
line_packs = None
if USE_SYNTHETIC_LINES:
    from codes.synthetic_scan_line_generator_v2 import generate_synthetic_scan_line
    from codes.dt_static_dynamic_calibration_scaffold import make_synthetic_lines_from_map

    synthetic_lines, line_packs = make_synthetic_lines_from_map(
        h_map,
        generate_synthetic_scan_line=generate_synthetic_scan_line,
        n_lines=int(N_FIT_LINES),
        n_pixels=traces_height.shape[-1],
        dx_hat=1.0,
        scan_speed_hat_for_generation=1.0,
        mode="block",                   # spatial-correlation-preserving
        direction="x",
        block_len=32,
        preserve_max_abs_dh=True,
        slope_limit_mode="rescale",
        seed0=RANDOM_SEED,
    )
    synthetic_lines = [_prepare_fit_line(ln) for ln in synthetic_lines]

fit_lines = synthetic_lines if USE_SYNTHETIC_LINES else h_line_exp

# Median experimental trace as a visual reference (centered for plotting).
h_line_reference = np.nanmedian(
    traces_height[:, :, :, :, 0, :].reshape(-1, traces_height.shape[-1]),
    axis=0,
)
# Match reference RMS too so the overlay is visually fair.
h_line_reference = _prepare_fit_line(h_line_reference)


# ---- Sanity check: report RMS of the prepared input lines ----
fit_rms = np.array([_exp_trace_rms(ln) for ln in fit_lines], dtype=float)
print(f"input-line RMS after match: median = {np.nanmedian(fit_rms):.3f} nm,  "
      f"5/95 pct = {np.nanpercentile(fit_rms, 5):.3f} / "
      f"{np.nanpercentile(fit_rms, 95):.3f} nm  (target = "
      f"{TARGET_RMS_NM:.3f} nm)")


# ---- Visual confirmation ----
fig, ax = plt.subplots(1, 1, figsize=(5.0, 2.4), constrained_layout=True)
for line in fit_lines[: min(12, len(fit_lines))]:
    ax.plot(line - np.nanmedian(line), lw=0.7, alpha=0.55, color="tab:blue")
ax.plot(h_line_reference - np.nanmedian(h_line_reference),
        color="black", lw=1.6, label="median exp. trace")
ax.set_xlabel("pixel")
ax.set_ylabel("height, centered (nm)")
ax.set_title(
    f"PI-fitting scan lines  (n = {len(fit_lines)}, "
    f"{'synthetic - block' if USE_SYNTHETIC_LINES else 'random topo rows'}, "
    f"detrended + RMS-matched to {TARGET_RMS_NM:.2f} nm)",
    fontsize=9,
)
ax.legend(frameon=False, fontsize=8)
print(f"prepared {len(fit_lines)} scan lines for PI fitting "
      f"(USE_SYNTHETIC_LINES = {USE_SYNTHETIC_LINES})")
print(f"row ids sample: {row_ids[:10]}{' ...' if len(row_ids) > 10 else ''}")


## 4. Physics-Guided PI Fitting

This stage fits local PI values on a subset of informative experimental conditions, then trains a GP map from `(scan speed, drive, setpoint, experimental I gain)` to DT controller `(P, I)`.


In [ ]:
from codes.dt_static_dynamic_calibration_scaffold_v3 import ScannerRunConfig
import codes.dt_gp_static_calibration_v3 as gpcal
importlib.reload(gpcal)

GPSubsetCalibrationConfig = gpcal.GPSubsetCalibrationConfig
fit_gp_static_controller_calibration = gpcal.fit_gp_static_controller_calibration
predict_PI_for_full_grid = gpcal.predict_PI_for_full_grid

scanner_cfg = ScannerRunConfig(
    dx_hat=1.0,
    n_substeps=50,
    tau_A=None,
    tau_phi=None,
    tau_z=None,
    z_rate_limit_hat=1e6,
    d_init_hat=0.0,
    A_meas_init_hat=1.0,
    phi_meas_init_deg=120.0,
    T_I=3e-1,
    h_smooth_sigma_px=0.0,
    fd_n_d=4096,
    use_fast=True,
)

gp_cfg = GPSubsetCalibrationConfig(
    n_train=int(CFG["n_train"]),
    n_test=int(CFG["n_test"]),
    seed=RANDOM_SEED,
    n_lines_train=int(CFG["n_lines_train"]),
    n_lines_test=int(CFG["n_lines_test"]),
    trim=10,
    filter_uninformative=True,
    min_mad=0.02,
    min_range=0.10,
    min_grad_mad=0.002,
    min_nonzero_frac=0.05,
    bounds_logPI=((-3, 3), (-3, 3)),
    use_grid_init=True,
    n_grid_P=int(CFG["n_grid_P"]),
    n_grid_I=int(CFG["n_grid_I"]),
    do_local_refine=True,
    maxiter_local=int(CFG["maxiter_local"]),
    w_trrt=100.0,
    w_grad_mean=10.0,
    w_grad_std=5.0,
    w_grad_p95=0.0,
    w_height_mad=0.0,
    w_height_range=0.0,
)

physics_cache = CACHE_DIR / f"physics_guided_gp_fit_{CACHE_TAG}.joblib"
if physics_cache.exists() and not REFIT_PHYSICS_GUIDED:
    gp_fit = load(physics_cache)
    print("loaded", physics_cache)
else:
    t0 = time.time()
    gp_fit = fit_gp_static_controller_calibration(
        scanner,
        traces_height,
        fit_lines,
        scan_speeds_hat=scan_rate,
        drives_nm=drive_exp,
        setpoints=setpoint_exp,
        i_gains_exp=igain_exp,
        scanner_cfg=scanner_cfg,
        cfg=gp_cfg,
        verbose=True,
    )
    dump(gp_fit, physics_cache)
    print("saved", physics_cache, "elapsed min", (time.time() - t0) / 60)

print("Physics-guided test summary:")
print(gp_fit["test_summary"])
print("Rejected/uninformative:", len(gp_fit.get("rejected_conditions", [])))


In [ ]:
P_grid, I_grid, P_std_grid, I_std_grid, pi_pred_rows = predict_PI_for_full_grid(
    gp_fit["gp_pack"],
    traces_height,
    scan_speeds_hat=scan_rate,
    drives_nm=drive_exp,
    setpoints=setpoint_exp,
    i_gains_exp=igain_exp,
)

pi_artifact = {
    "P_grid": P_grid,
    "I_grid": I_grid,
    "P_std_grid": P_std_grid,
    "I_std_grid": I_std_grid,
    "pred_rows": pi_pred_rows,
    "scan_rate": scan_rate,
    "drive_exp": drive_exp,
    "setpoint_exp": setpoint_exp,
    "igain_exp": igain_exp,
    "gp_fit": gp_fit,
}
dump(pi_artifact, CACHE_DIR / f"physics_guided_PI_grid_{CACHE_TAG}.joblib")

print("P range:", np.nanpercentile(P_grid, [1, 50, 99]))
print("I range:", np.nanpercentile(I_grid, [1, 50, 99]))


In [ ]:
def center_line(y):
    y = np.asarray(y, dtype=float).ravel()
    return y - np.nanmedian(y)


def robust_rmse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if np.sum(mask) == 0:
        return np.nan
    return float(np.sqrt(np.nanmean((a[mask] - b[mask]) ** 2)))


def simulate_condition(cond, *, h_line=None, P_override=None, I_override=None):
    si, di, spi, gi = [int(v) for v in cond]
    P = float(P_grid[si, di, spi, gi] if P_override is None else P_override)
    I = float(I_grid[si, di, spi, gi] if I_override is None else I_override)
    if h_line is None:
        h_line = h_line_reference
    fd_table = scanner.prepare_fd_table_for_drive(float(drive_exp[di]), n_d=scanner_cfg.fd_n_d)
    out = scanner.simulate_trace_retrace_substeps_fast(
        h_line,
        drive_nm=float(drive_exp[di]),
        setpoint=float(setpoint_exp[spi]),
        scan_speed_hat=float(scan_rate[si]),
        dx_hat=scanner_cfg.dx_hat,
        P=P,
        I=I,
        A_filter_order=None,
        tau_A=None,
        phi_filter_order=None,
        tau_phi=None,
        z_filter_order=None,
        tau_z=None,
        z_rate_limit_hat=scanner_cfg.z_rate_limit_hat,
        d_init_hat=scanner_cfg.d_init_hat,
        A_meas_init_hat=scanner_cfg.A_meas_init_hat,
        phi_meas_init_deg=scanner_cfg.phi_meas_init_deg,
        T_I=scanner_cfg.T_I,
        integ_clip=scanner_cfg.integ_clip,
        n_substeps=scanner_cfg.n_substeps,
        antiwindup=scanner_cfg.antiwindup,
        contact_A_frac=scanner_cfg.contact_A_frac,
        saturation_A_frac=scanner_cfg.saturation_A_frac,
        record_mode=scanner_cfg.record_mode,
        carry_state_to_retrace=scanner_cfg.carry_state_to_retrace,
        h_smooth_sigma_px=scanner_cfg.h_smooth_sigma_px,
        fd_table=fd_table,
        fd_n_d=scanner_cfg.fd_n_d,
    )
    return out


def plot_pi_maps_and_trace_examples(example_conds=None):
    if example_conds is None:
        # Pick three representative (si, di, spi, gi) tuples that are valid
        # for the (possibly gain-filtered) grid.
        n_speed_, n_drive_, n_setpoint_, n_gain_ = P_grid.shape
        gi_mid = min(2, n_gain_ - 1)
        gi_top = n_gain_ - 1
        example_conds = [
            (0,                            min(3, n_drive_ - 1), min(2, n_setpoint_ - 1), 0),
            (min(2, n_speed_ - 1),         min(6, n_drive_ - 1), min(4, n_setpoint_ - 1), gi_mid),
            (n_speed_ - 1,                 min(8, n_drive_ - 1), min(5, n_setpoint_ - 1), gi_top),
        ]

    selected_gain = int(np.argmin(np.abs(igain_exp - np.median(igain_exp))))
    selected_speed = int(np.argmin(np.abs(scan_rate - np.median(scan_rate))))

    fig = plt.figure(figsize=(7.2, 5.8), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[1.0, 1.0, 1.15])

    axP = fig.add_subplot(gs[0, 0])
    axI = fig.add_subplot(gs[0, 1])
    axLoss = fig.add_subplot(gs[0, 2])

    logP = np.log10(P_grid[selected_speed, :, :, selected_gain])
    logI = np.log10(I_grid[selected_speed, :, :, selected_gain])
    imP = axP.imshow(logP.T, origin="lower", aspect="auto", cmap="mako" if "mako" in plt.colormaps() else "viridis")
    imI = axI.imshow(logI.T, origin="lower", aspect="auto", cmap="rocket" if "rocket" in plt.colormaps() else "magma")
    for ax in [axP, axI]:
        ax.set_xticks(range(len(drive_exp)), [f"{v:.1f}" for v in drive_exp], rotation=45, ha="right")
        ax.set_yticks(range(len(setpoint_exp)), [f"{v:.1f}" for v in setpoint_exp])
        ax.set_xlabel("drive (nm)")
        ax.set_ylabel("setpoint")
    axP.set_title(f"log10 P, speed={scan_rate[selected_speed]:g}, Iexp={igain_exp[selected_gain]:g}")
    axI.set_title(f"log10 I, speed={scan_rate[selected_speed]:g}, Iexp={igain_exp[selected_gain]:g}")
    fig.colorbar(imP, ax=axP, fraction=0.046, pad=0.02)
    fig.colorbar(imI, ax=axI, fraction=0.046, pad=0.02)

    test_losses = [r["test_loss"] for r in gp_fit["test_rows"]]
    axLoss.hist(test_losses, bins=18, color="0.25", alpha=0.85)
    axLoss.axvline(np.nanmedian(test_losses), color="tab:red", lw=1.2)
    axLoss.set_xlabel("held-out static loss")
    axLoss.set_ylabel("count")
    axLoss.set_title("PI fit test losses")

    for j, cond in enumerate(example_conds):
        si, di, spi, gi = cond
        out = simulate_condition(cond)
        for direction, direction_name in enumerate(["trace", "retrace"]):
            ax = fig.add_subplot(gs[1 + direction, j])
            exp_line = center_line(traces_height[si, di, spi, gi, direction])
            sim_line = center_line(out[direction_name]["d_hat"])
            ax.plot(exp_line, lw=1.0, color="black", label="experiment")
            ax.plot(sim_line, lw=1.0, color="tab:blue", label="DT + fitted PI")
            ax.set_title(f"s={scan_rate[si]:g}, d={drive_exp[di]:.1f}, sp={setpoint_exp[spi]:.1f}, Ig={igain_exp[gi]:.0f}")
            ax.set_xlabel("pixel")
            ax.set_ylabel(f"{direction_name} h (nm)")
            ax.text(0.02, 0.93, f"RMSE {robust_rmse(exp_line, sim_line):.2f} nm", transform=ax.transAxes, va="top")
            if j == 0 and direction == 0:
                ax.legend(frameon=False, loc="best")

    return fig

fig_pi = plot_pi_maps_and_trace_examples()
save_figure(fig_pi, "figure_1_physics_guided_PI_and_trace_alignment")


## 5. Pure Data-Driven Trace Alignment

This section builds a row per scan condition and direction. Two models are trained on held-out condition splits:

- `direct`: predicts the experimental centered trace from controls and FD-prior features only.
- `residual`: predicts a correction to the PI-fitted DT trace, so `aligned = DT + residual`.

The residual model is the final DT-alignment layer; the direct model is a pure experimental-data baseline.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def fd_target_lookup_table(scanner, drives, setpoints, fd_n_d=4096):
    rows = {}
    for di, drv in enumerate(drives):
        fd = scanner.prepare_fd_table_for_drive(float(drv), n_d=fd_n_d)
        A0 = float(fd["A0"])
        for spi, sp in enumerate(setpoints):
            target = float(sp) * A0
            idx = int(np.nanargmin(np.abs(fd["A_grid"] - target)))
            rows[(di, spi)] = {
                "A_fd": float(fd["A_grid"][idx]),
                "phi_fd": float(fd["phi_grid"][idx]),
                "d_fd": float(fd["d_grid"][idx]),
                "A0_fd": A0,
            }
    return rows


def line_stats(y):
    y = np.asarray(y, dtype=float).ravel()
    dy = np.diff(y)
    return {
        "line_std": float(np.nanstd(y)),
        "line_iqr": float(np.nanpercentile(y, 75) - np.nanpercentile(y, 25)),
        "grad_mad": float(1.4826 * np.nanmedian(np.abs(dy - np.nanmedian(dy)))) if dy.size else np.nan,
    }


def build_condition_records():
    fd_lookup = fd_target_lookup_table(scanner, drive_exp, setpoint_exp, fd_n_d=2048)
    records = []
    Y_exp = []
    Y_dt = []

    all_conds = [(si, di, spi, gi)
                 for si in range(len(scan_rate))
                 for di in range(len(drive_exp))
                 for spi in range(len(setpoint_exp))
                 for gi in range(len(igain_exp))]
    max_conditions = CFG["dd_max_conditions"]
    if max_conditions is not None and max_conditions < len(all_conds):
        rng = np.random.default_rng(RANDOM_SEED + 10)
        keep = sorted(rng.choice(len(all_conds), size=int(max_conditions), replace=False))
        all_conds = [all_conds[i] for i in keep]

    fd_table_cache = {}
    for n, cond in enumerate(all_conds, start=1):
        si, di, spi, gi = cond
        if di not in fd_table_cache:
            fd_table_cache[di] = scanner.prepare_fd_table_for_drive(float(drive_exp[di]), n_d=scanner_cfg.fd_n_d)
        out = scanner.simulate_trace_retrace_substeps_fast(
            h_line_reference,
            drive_nm=float(drive_exp[di]),
            setpoint=float(setpoint_exp[spi]),
            scan_speed_hat=float(scan_rate[si]),
            dx_hat=scanner_cfg.dx_hat,
            P=float(P_grid[si, di, spi, gi]),
            I=float(I_grid[si, di, spi, gi]),
            A_filter_order=None,
            tau_A=None,
            phi_filter_order=None,
            tau_phi=None,
            z_filter_order=None,
            tau_z=None,
            z_rate_limit_hat=scanner_cfg.z_rate_limit_hat,
            d_init_hat=scanner_cfg.d_init_hat,
            A_meas_init_hat=scanner_cfg.A_meas_init_hat,
            phi_meas_init_deg=scanner_cfg.phi_meas_init_deg,
            T_I=scanner_cfg.T_I,
            integ_clip=scanner_cfg.integ_clip,
            n_substeps=scanner_cfg.n_substeps,
            antiwindup=scanner_cfg.antiwindup,
            contact_A_frac=scanner_cfg.contact_A_frac,
            saturation_A_frac=scanner_cfg.saturation_A_frac,
            record_mode=scanner_cfg.record_mode,
            carry_state_to_retrace=scanner_cfg.carry_state_to_retrace,
            h_smooth_sigma_px=scanner_cfg.h_smooth_sigma_px,
            fd_table=fd_table_cache[di],
            fd_n_d=scanner_cfg.fd_n_d,
        )
        for direction, direction_name, direction_sign in [(0, "trace", -1.0), (1, "retrace", 1.0)]:
            exp_line = center_line(traces_height[si, di, spi, gi, direction])
            dt_line = center_line(out[direction_name]["d_hat"])
            fdrow = fd_lookup[(di, spi)]
            stats = line_stats(dt_line)
            records.append({
                "cond_key": f"{si}_{di}_{spi}_{gi}",
                "si": si,
                "di": di,
                "spi": spi,
                "gi": gi,
                "direction": direction_name,
                "direction_sign": direction_sign,
                "scan_speed_hat": float(scan_rate[si]),
                "log_scan_speed": float(np.log10(max(scan_rate[si], 1e-12))),
                "drive_nm": float(drive_exp[di]),
                "setpoint": float(setpoint_exp[spi]),
                "I_exp": float(igain_exp[gi]),
                "log_I_exp": float(np.log10(max(igain_exp[gi], 1e-12))),
                "P_dt": float(P_grid[si, di, spi, gi]),
                "I_dt": float(I_grid[si, di, spi, gi]),
                "log_P_dt": float(np.log10(max(P_grid[si, di, spi, gi], 1e-12))),
                "log_I_dt": float(np.log10(max(I_grid[si, di, spi, gi], 1e-12))),
                **fdrow,
                **stats,
                "dt_rmse": robust_rmse(exp_line, dt_line),
            })
            Y_exp.append(exp_line)
            Y_dt.append(dt_line)
        if n % 100 == 0 or n == len(all_conds):
            print(f"simulated {n}/{len(all_conds)} conditions for data-driven alignment")

    return pd.DataFrame(records), np.asarray(Y_exp), np.asarray(Y_dt)

records_cache = CACHE_DIR / f"data_driven_records_{CACHE_TAG}.joblib"
if records_cache.exists() and not REFIT_DATA_DRIVEN:
    rec_pack = load(records_cache)
    record_df = rec_pack["record_df"]
    Y_exp = rec_pack["Y_exp"]
    Y_dt = rec_pack["Y_dt"]
    print("loaded", records_cache)
else:
    t0 = time.time()
    record_df, Y_exp, Y_dt = build_condition_records()
    dump({"record_df": record_df, "Y_exp": Y_exp, "Y_dt": Y_dt}, records_cache)
    print("saved", records_cache, "elapsed min", (time.time() - t0) / 60)

record_df.head()


In [ ]:
def split_by_condition(record_df, test_size=0.25, seed=RANDOM_SEED):
    keys = np.array(sorted(record_df["cond_key"].unique()))
    train_keys, test_keys = train_test_split(keys, test_size=test_size, random_state=seed)
    train_mask = record_df["cond_key"].isin(train_keys).to_numpy()
    test_mask = record_df["cond_key"].isin(test_keys).to_numpy()
    return train_mask, test_mask, train_keys, test_keys


def fit_pca_tree_model(X_train, Y_train, *, n_components=20, n_estimators=450, seed=RANDOM_SEED):
    n_components = int(min(n_components, Y_train.shape[0] - 1, Y_train.shape[1]))
    pca = PCA(n_components=n_components, random_state=seed)
    scores = pca.fit_transform(Y_train)
    reg = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        ExtraTreesRegressor(
            n_estimators=int(n_estimators),
            min_samples_leaf=2,
            max_features=1.0,
            random_state=seed,
            n_jobs=-1,
        ),
    )
    reg.fit(X_train, scores)
    return {"pca": pca, "reg": reg}


def predict_pca_tree_model(model, X):
    scores = model["reg"].predict(X)
    return model["pca"].inverse_transform(scores)


def line_metrics(Y_true, Y_pred):
    err = Y_pred - Y_true
    rmse_per_line = np.sqrt(np.nanmean(err ** 2, axis=1))
    mae_per_line = np.nanmean(np.abs(err), axis=1)
    return {
        "rmse_median": float(np.nanmedian(rmse_per_line)),
        "rmse_mean": float(np.nanmean(rmse_per_line)),
        "mae_median": float(np.nanmedian(mae_per_line)),
        "mae_mean": float(np.nanmean(mae_per_line)),
    }

base_feature_cols = [
    "log_scan_speed", "drive_nm", "setpoint", "log_I_exp", "direction_sign",
    "A_fd", "phi_fd", "d_fd", "A0_fd",
]
residual_feature_cols = base_feature_cols + [
    "log_P_dt", "log_I_dt", "line_std", "line_iqr", "grad_mad",
]

train_mask, test_mask, train_keys, test_keys = split_by_condition(record_df, test_size=0.25)

X_direct_train = record_df.loc[train_mask, base_feature_cols].to_numpy(float)
X_direct_test = record_df.loc[test_mask, base_feature_cols].to_numpy(float)
X_resid_train = record_df.loc[train_mask, residual_feature_cols].to_numpy(float)
X_resid_test = record_df.loc[test_mask, residual_feature_cols].to_numpy(float)

Y_resid = Y_exp - Y_dt

direct_model = fit_pca_tree_model(
    X_direct_train,
    Y_exp[train_mask],
    n_components=24,
    n_estimators=CFG["n_estimators"],
    seed=RANDOM_SEED,
)
residual_model = fit_pca_tree_model(
    X_resid_train,
    Y_resid[train_mask],
    n_components=24,
    n_estimators=CFG["n_estimators"],
    seed=RANDOM_SEED + 1,
)

Y_direct_test = predict_pca_tree_model(direct_model, X_direct_test)
Y_resid_pred_test = predict_pca_tree_model(residual_model, X_resid_test)
Y_aligned_test = Y_dt[test_mask] + Y_resid_pred_test

metric_rows = [
    {"model": "DT + fitted PI", **line_metrics(Y_exp[test_mask], Y_dt[test_mask])},
    {"model": "direct data-driven", **line_metrics(Y_exp[test_mask], Y_direct_test)},
    {"model": "DT + data-driven residual", **line_metrics(Y_exp[test_mask], Y_aligned_test)},
]
metric_df = pd.DataFrame(metric_rows)
display(metric_df)

data_driven_pack = {
    "record_df": record_df,
    "Y_exp": Y_exp,
    "Y_dt": Y_dt,
    "train_mask": train_mask,
    "test_mask": test_mask,
    "train_keys": train_keys,
    "test_keys": test_keys,
    "direct_model": direct_model,
    "residual_model": residual_model,
    "Y_direct_test": Y_direct_test,
    "Y_aligned_test": Y_aligned_test,
    "metric_df": metric_df,
    "base_feature_cols": base_feature_cols,
    "residual_feature_cols": residual_feature_cols,
}
dump(data_driven_pack, CACHE_DIR / f"data_driven_alignment_{CACHE_TAG}.joblib")


In [ ]:
def plot_data_driven_alignment_figure():
    test_indices = np.where(test_mask)[0]
    test_df = record_df.loc[test_mask].reset_index(drop=True)
    dt_rmse = np.sqrt(np.nanmean((Y_dt[test_mask] - Y_exp[test_mask]) ** 2, axis=1))
    direct_rmse = np.sqrt(np.nanmean((Y_direct_test - Y_exp[test_mask]) ** 2, axis=1))
    aligned_rmse = np.sqrt(np.nanmean((Y_aligned_test - Y_exp[test_mask]) ** 2, axis=1))

    order = np.argsort(aligned_rmse)
    pick_local = [order[0], order[len(order)//2], order[-1]]

    fig = plt.figure(figsize=(7.2, 5.9), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[0.95, 1.1, 1.1])

    ax_bar = fig.add_subplot(gs[0, 0])
    colors = ["0.55", "tab:green", "tab:orange"]
    ax_bar.bar(metric_df["model"], metric_df["rmse_median"], color=colors, width=0.72)
    ax_bar.set_ylabel("median RMSE (nm)")
    ax_bar.set_title("held-out trace error")
    ax_bar.tick_params(axis="x", rotation=30)
    for tick in ax_bar.get_xticklabels():
        tick.set_ha("right")

    ax_hist = fig.add_subplot(gs[0, 1])
    bins = np.linspace(0, np.nanpercentile(dt_rmse, 98), 28)
    ax_hist.hist(dt_rmse, bins=bins, histtype="step", lw=1.4, color="0.35", label="DT")
    ax_hist.hist(direct_rmse, bins=bins, histtype="step", lw=1.4, color="tab:green", label="direct")
    ax_hist.hist(aligned_rmse, bins=bins, histtype="step", lw=1.4, color="tab:orange", label="DT + residual")
    ax_hist.set_xlabel("line RMSE (nm)")
    ax_hist.set_ylabel("count")
    ax_hist.set_title("error distribution")
    ax_hist.legend(frameon=False)

    ax_scatter = fig.add_subplot(gs[0, 2])
    ax_scatter.scatter(dt_rmse, aligned_rmse, s=8, alpha=0.45, color="tab:orange", edgecolor="none")
    lim = [0, max(np.nanpercentile(dt_rmse, 98), np.nanpercentile(aligned_rmse, 98))]
    ax_scatter.plot(lim, lim, color="0.25", lw=0.8, ls="--")
    ax_scatter.set_xlim(lim)
    ax_scatter.set_ylim(lim)
    ax_scatter.set_xlabel("DT RMSE (nm)")
    ax_scatter.set_ylabel("aligned RMSE (nm)")
    ax_scatter.set_title("per-line improvement")

    for col, loc in enumerate(pick_local):
        global_idx = test_indices[loc]
        row = test_df.iloc[loc]
        exp_line = Y_exp[global_idx]
        dt_line = Y_dt[global_idx]
        direct_line = Y_direct_test[loc]
        aligned_line = Y_aligned_test[loc]

        ax_line = fig.add_subplot(gs[1, col])
        ax_line.plot(exp_line, color="black", lw=1.0, label="experiment")
        ax_line.plot(dt_line, color="0.55", lw=0.9, label="DT")
        ax_line.plot(aligned_line, color="tab:orange", lw=1.0, label="DT + residual")
        ax_line.plot(direct_line, color="tab:green", lw=0.9, alpha=0.85, label="direct")
        ax_line.set_title(
            f"{row.direction}: s={row.scan_speed_hat:g}, d={row.drive_nm:.1f}, sp={row.setpoint:.1f}, Ig={row.I_exp:.0f}"
        )
        ax_line.set_xlabel("pixel")
        ax_line.set_ylabel("height (nm)")
        ax_line.text(
            0.02, 0.94,
            f"DT {dt_rmse[loc]:.2f}\naligned {aligned_rmse[loc]:.2f}",
            transform=ax_line.transAxes,
            va="top",
            ha="left",
            fontsize=7,
        )
        if col == 0:
            ax_line.legend(frameon=False, loc="best")

        ax_err = fig.add_subplot(gs[2, col])
        ax_err.plot(dt_line - exp_line, color="0.55", lw=0.9, label="DT error")
        ax_err.plot(aligned_line - exp_line, color="tab:orange", lw=1.0, label="aligned error")
        ax_err.axhline(0, color="0.2", lw=0.6)
        ax_err.set_xlabel("pixel")
        ax_err.set_ylabel("residual (nm)")
        ax_err.set_title("error traces")

    return fig

fig_dd = plot_data_driven_alignment_figure()
save_figure(fig_dd, "figure_2_data_driven_DT_trace_alignment")


## 6. Held-Out Demo: Fit Quality On Unseen Experimental Scan Lines

The PI test set inside `gp_fit` (30 conditions) and the data-driven test mask
each guard one of the two pipelines, but they overlap. To show fit quality on
scan lines that **neither** pipeline ever saw, we intersect the two held-out
sets and pick a representative slice of the resulting truly-unseen
conditions. For each, the experimental height trace is overlaid with the
DT trace produced at the GP-predicted (P, I) and the direct data-driven
trace prediction.


In [ ]:
# Truly held-out: experimental scan lines never used to train EITHER pipeline.
gp_train_set = set(tuple(int(v) for v in c) for c in gp_fit["train_conds"])
test_indices = np.where(test_mask)[0]
test_df_local = record_df.loc[test_mask].reset_index(drop=True)

eligible_local = []
for k, idx in enumerate(test_indices):
    row = record_df.iloc[idx]
    cond = (int(row["si"]), int(row["di"]), int(row["spi"]), int(row["gi"]))
    if cond in gp_train_set:
        continue
    eligible_local.append(k)
eligible_local = np.array(eligible_local, dtype=int)
print(f"truly held-out from BOTH pipelines : {eligible_local.size} of {test_mask.sum()} dd test rows")

# Per-line RMSE on the held-out subset for the two pipelines.
dt_rmse_all      = np.sqrt(np.nanmean((Y_dt[test_mask]    - Y_exp[test_mask]) ** 2, axis=1))
direct_rmse_all  = np.sqrt(np.nanmean((Y_direct_test     - Y_exp[test_mask]) ** 2, axis=1))
aligned_rmse_all = np.sqrt(np.nanmean((Y_aligned_test    - Y_exp[test_mask]) ** 2, axis=1))

ho_dt      = dt_rmse_all     [eligible_local]
ho_direct  = direct_rmse_all [eligible_local]
ho_aligned = aligned_rmse_all[eligible_local]

print(f"median RMSE (nm) on the truly-unseen set: "
      f"phys={np.nanmedian(ho_dt):.3f}, "
      f"data-direct={np.nanmedian(ho_direct):.3f}, "
      f"data-residual={np.nanmedian(ho_aligned):.3f}")
print(f"mean   RMSE (nm) on the truly-unseen set: "
      f"phys={np.nanmean(ho_dt):.3f}, "
      f"data-direct={np.nanmean(ho_direct):.3f}, "
      f"data-residual={np.nanmean(ho_aligned):.3f}")


In [ ]:
# Pick 12 representative held-out conditions, RMSE-quantile-spaced in the
# 5th-85th percentile range (avoid trivial cases and the long outlier tail).
n_pick = 12
ranked_by_rmse = eligible_local[np.argsort(ho_dt)]
quantiles = np.linspace(0.05, 0.85, n_pick)
pick_idx = np.unique(
    [ranked_by_rmse[int(q * (len(ranked_by_rmse) - 1))] for q in quantiles]
)
# Pad in case dedupe shrank the list.
while len(pick_idx) < n_pick:
    extra = ranked_by_rmse[len(pick_idx) % len(ranked_by_rmse)]
    pick_idx = np.append(pick_idx, extra)
pick_idx = pick_idx[:n_pick]
print("picked dd-test rows:", pick_idx)

# Render the 3 x 4 demo grid.
fig = plt.figure(figsize=(12.6, 7.2), constrained_layout=True)
gs = fig.add_gridspec(3, 4, hspace=0.05, wspace=0.05)

for k, loc in enumerate(pick_idx):
    row = test_df_local.iloc[int(loc)]
    global_idx = test_indices[int(loc)]

    exp_line    = Y_exp[global_idx]
    dt_line     = Y_dt [global_idx]
    direct_line = Y_direct_test[int(loc)]

    ax = fig.add_subplot(gs[k // 4, k % 4])
    ax.plot(exp_line,    color="black",     lw=1.1, label="experiment")
    ax.plot(dt_line,     color="tab:blue",  lw=1.1, alpha=0.9, label="DT + fitted PI")
    ax.plot(direct_line, color="tab:green", lw=1.1, alpha=0.9, label="data-driven")

    ax.set_title(
        f"s={row.scan_speed_hat:g}  d={row.drive_nm:.1f}  "
        f"sp={row.setpoint:.1f}  Ig={row.I_exp:.0f}  ({row.direction[:2]})",
        fontsize=7.5,
    )
    ax.text(
        0.02, 0.96,
        f"phys {ho_dt[np.where(eligible_local == loc)[0][0]]:.2f}\n"
        f"data {ho_direct[np.where(eligible_local == loc)[0][0]]:.2f}",
        transform=ax.transAxes, va="top", ha="left", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                  alpha=0.8, edgecolor="0.75"),
    )
    if k == 0:
        ax.legend(loc="lower right", fontsize=6.5, frameon=False)

    ax.set_xlabel("pixel" if k // 4 == 2 else "")
    ax.set_ylabel("height (nm)" if k % 4 == 0 else "")
    ax.tick_params(axis="both", labelsize=7)

# Summary footnote at the bottom-right.
fig.text(
    0.99, -0.02,
    f"truly held-out (no train on either pipeline): "
    f"n={eligible_local.size}, "
    f"median RMSE  phys={np.nanmedian(ho_dt):.2f} nm  data={np.nanmedian(ho_direct):.2f} nm",
    ha="right", va="top", fontsize=7, color="0.35",
)

save_figure(fig, "figure_3_held_out_demo")
fig


## 7.  Full Fit With a Random Batch of Topo Scan Lines

To check that the calibrated DT is not over-fit to the specific topo
rows used in Sections 4-6, we now re-run the **entire** fitting
pipeline — local PI fits, GP regression, full-grid PI prediction, and
data-driven trace alignment — on an independently-drawn random batch
of measured rows from the same topo image.  A different
`RANDOM_SEED_RAND` and a separate cache tag keep this run isolated
from the main caches.


In [ ]:
# ---------------------------------------------------------------
# Set up the random-scan-line run.
# ---------------------------------------------------------------
import time as _time
import copy

RANDOM_SEED_RAND   = 1234
N_RAND_LINES       = max(int(CFG["n_lines_test"]) * 4, 16)
CACHE_TAG_RAND     = f"{CACHE_TAG}_rand{RANDOM_SEED_RAND}"

rng_rand = np.random.default_rng(RANDOM_SEED_RAND)
row_ids_rand = np.sort(
    rng_rand.choice(h_map.shape[0], size=int(N_RAND_LINES), replace=False)
)
h_line_rand = np.asarray(
    [_prepare_fit_line(h_map[r, :traces_height.shape[-1]])
     for r in row_ids_rand],
    dtype=float,
)

fig, ax = plt.subplots(1, 1, figsize=(5.0, 2.2), constrained_layout=True)
for line in h_line_rand[:12]:
    ax.plot(line - np.nanmedian(line), lw=0.7, alpha=0.55, color="tab:orange")
ax.plot(h_line_reference - np.nanmedian(h_line_reference),
        color="black", lw=1.6, label="median exp. trace")
ax.set_xlabel("pixel")
ax.set_ylabel("height, centered (nm)")
ax.set_title(
    f"random-fit input scan lines  (seed = {RANDOM_SEED_RAND}, "
    f"n = {len(h_line_rand)})",
    fontsize=9,
)
ax.legend(frameon=False, fontsize=8)
print(f"random batch: {len(h_line_rand)} rows, sample ids "
      f"{row_ids_rand[:10]}{' ...' if len(row_ids_rand) > 10 else ''}")


In [ ]:
# ---------------------------------------------------------------
# Re-run the GP fit on the random batch.  Cached separately.
# ---------------------------------------------------------------
REFIT_PHYSICS_GUIDED_RAND = False
physics_cache_rand = CACHE_DIR / f"physics_guided_gp_fit_{CACHE_TAG_RAND}.joblib"

# Use the same gp_cfg as the main fit (same n_train, n_test, weights, etc.)
gp_cfg_rand = copy.deepcopy(gp_cfg)
gp_cfg_rand.seed = RANDOM_SEED_RAND       # different oracle-fit selection too

if physics_cache_rand.exists() and not REFIT_PHYSICS_GUIDED_RAND:
    gp_fit_rand = load(physics_cache_rand)
    print("loaded", physics_cache_rand)
else:
    t0 = _time.time()
    gp_fit_rand = fit_gp_static_controller_calibration(
        scanner,
        traces_height,
        h_line_rand,
        scan_speeds_hat=scan_rate,
        drives_nm=drive_exp,
        setpoints=setpoint_exp,
        i_gains_exp=igain_exp,
        scanner_cfg=scanner_cfg,
        cfg=gp_cfg_rand,
        verbose=True,
    )
    dump(gp_fit_rand, physics_cache_rand)
    print(f"saved {physics_cache_rand}  elapsed min = "
          f"{(_time.time() - t0) / 60:.2f}")

print("random-batch GP fit test_summary:")
print(gp_fit_rand["test_summary"])

# Full-grid PI prediction with the random-batch GP map.
P_grid_rand, I_grid_rand, P_std_rand, I_std_rand, pi_pred_rand = predict_PI_for_full_grid(
    gp_fit_rand["gp_pack"],
    traces_height,
    scan_speeds_hat=scan_rate,
    drives_nm=drive_exp,
    setpoints=setpoint_exp,
    i_gains_exp=igain_exp,
)
print(f"random-batch P range : {np.nanpercentile(P_grid_rand, [1, 50, 99])}")
print(f"random-batch I range : {np.nanpercentile(I_grid_rand, [1, 50, 99])}")


In [ ]:
# ---------------------------------------------------------------
# Compare main-batch vs random-batch GP + DT performance on the
# main-batch held-out test conditions.  Lower is better.
# ---------------------------------------------------------------
def _rmse_on_test_rows(P_grid_x, I_grid_x, test_rows, h_line_input):
    """RMSE of (DT trace at predicted P,I) vs exp trace on the test rows."""
    rmses = []
    for r in test_rows:
        si, di, spi, gi = tuple(int(v) for v in r["cond"])
        try:
            out = simulate_condition(
                (si, di, spi, gi),
                h_line=h_line_reference,
                P_override=float(P_grid_x[si, di, spi, gi]),
                I_override=float(I_grid_x[si, di, spi, gi]),
            )
            sim_tr = np.asarray(out["trace"]["d_hat"], dtype=float)
            exp_tr = traces_height[si, di, spi, gi, 0]
            n = min(len(sim_tr), len(exp_tr))
            st = sim_tr[:n] - np.nanmean(sim_tr[:n])
            et = exp_tr[:n] - np.nanmean(exp_tr[:n])
            rmses.append(float(np.sqrt(np.nanmean((st - et) ** 2))))
        except Exception:
            rmses.append(np.nan)
    return np.array(rmses, dtype=float)


# Use the test rows from the MAIN GP fit so we compare two predictors on
# the same conditions.
test_rows_main = gp_fit.get("test_rows", [])
rmse_main = _rmse_on_test_rows(P_grid, I_grid, test_rows_main, h_line_exp)
rmse_rand = _rmse_on_test_rows(P_grid_rand, I_grid_rand, test_rows_main, h_line_rand)

print(f"held-out test set  (n = {len(rmse_main)} from main-fit test rows)")
print(f"  main-batch fit   :  median RMSE = "
      f"{np.nanmedian(rmse_main):.3f} nm   mean = {np.nanmean(rmse_main):.3f} nm")
print(f"  random-batch fit :  median RMSE = "
      f"{np.nanmedian(rmse_rand):.3f} nm   mean = {np.nanmean(rmse_rand):.3f} nm")

# Comparison figure
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4), constrained_layout=True)

# (a) RMSE bars
ax = axes[0]
labels = ["main batch", "random batch"]
vals   = [np.nanmedian(rmse_main), np.nanmedian(rmse_rand)]
colors = ["tab:blue", "tab:orange"]
bars = ax.bar(labels, vals, color=colors, alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.05, f"{v:.2f} nm",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("median held-out RMSE (nm)")
ax.set_title("(a) controller fit robustness")

# (b) per-line scatter
ax = axes[1]
lim = [0, max(np.nanpercentile(rmse_main, 98),
              np.nanpercentile(rmse_rand, 98), 1.0)]
ax.plot(lim, lim, "k:", lw=0.8)
ax.scatter(rmse_main, rmse_rand, s=18, alpha=0.7, color="tab:purple",
            edgecolor="k", lw=0.3)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("RMSE — main batch (nm)")
ax.set_ylabel("RMSE — random batch (nm)")
ax.set_title("(b) per-condition RMSE")

# (c) PI map difference
ax = axes[2]
selected_speed = int(np.argmin(np.abs(scan_rate - np.median(scan_rate))))
selected_gain  = int(np.argmin(np.abs(igain_exp - np.median(igain_exp))))
dlogP = (np.log10(P_grid_rand[selected_speed, :, :, selected_gain])
         - np.log10(P_grid     [selected_speed, :, :, selected_gain]))
im = ax.imshow(dlogP.T, origin="lower", aspect="auto",
               cmap="RdBu_r", vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(drive_exp)),
              [f"{v:.1f}" for v in drive_exp], rotation=45, ha="right",
              fontsize=7)
ax.set_yticks(range(len(setpoint_exp)),
              [f"{v:.1f}" for v in setpoint_exp], fontsize=7)
ax.set_xlabel("drive (nm)")
ax.set_ylabel("setpoint")
ax.set_title(f"(c) Δ log$_{{10}}$ P  (random − main)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

save_figure(fig, "figure_4_random_scan_line_fit")


## 8. Saved Outputs

Primary outputs:

- `output/dt_controller_fit_figures/figure_1_physics_guided_PI_and_trace_alignment.{png,pdf}`
- `output/dt_controller_fit_figures/figure_2_data_driven_DT_trace_alignment.{png,pdf}`
- `output/dt_controller_fit_figures/figure_3_held_out_demo.{png,pdf}`
- `output/dt_controller_fit_figures/figure_4_random_scan_line_fit.{png,pdf}`
- `calibration_cache/dt_controller_fit/physics_guided_gp_fit_<tag>.joblib`
- `calibration_cache/dt_controller_fit/data_driven_alignment_<tag>.joblib`


In [ ]:
print("Primary figure files:")
for path in sorted(FIG_DIR.glob("figure_*")):
    print(path)
print("\nCache files:")
for path in sorted(CACHE_DIR.glob(f"*{FIT_PROFILE}*")):
    print(path)


## Nature-style manuscript figure export

This final cell exports the manuscript figure package with the shared Nature-style layout contract: 183 mm print width, white background, lowercase panel labels, and high-DPI PNG/TIFF/PDF outputs under `output/nature_figures`.

In [ ]:
from pathlib import Path
import importlib
import codes.nature_style_outputs as nature_style_outputs

importlib.reload(nature_style_outputs)
project_root = Path.cwd().resolve()
nature_figures = nature_style_outputs.build_controller_fit_figures(project_root)
for figure_name, outputs in nature_figures.items():
    print(figure_name)
    for fmt, path in outputs.items():
        print(f"  {fmt}: {path}")


## 9.  Nature-style Figures 4 & 5

Two publication-quality figures, mirroring the calibration-grating workflow, but driven by the cached PI grid and data-driven artifacts from sections 4 & 5:

* **Figure 4** — Synthetic vs experimental $h_{\mathrm{truth}}$, GP-fitted PI loss surface, PI map generalisation, plus DT-vs-experiment overlays for optimal and sub-optimal control settings.
* **Figure 5** — Held-out scan-quality prediction (trace–retrace RMSE) for three controller models: PI fitting, PI + data-driven residual, and pure data-driven.

Pipeline:
1. Run `codes/expscan_build_predictions.py` — reads the cached PI grid and the data-driven alignment artifacts and writes `/tmp/expscan_predictions.npz`.
2. Save a plot-only `figure_45_expscan_data_bundle.npz` next to the figures so the figures can be redrawn in any other notebook without re-running the GP or the data-driven training.
3. Render Figures 4 & 5 from the bundle.


In [ ]:
# 9.1  Build the prediction cache.  Reads the cached PI + data-driven joblibs
#       and writes /tmp/expscan_predictions.npz.
import subprocess, sys
subprocess.check_call([sys.executable, str(PROJECT_ROOT / "codes/expscan_build_predictions.py"),
                       "--tag", CACHE_TAG])


In [ ]:
# 9.2  Save a plot-only data bundle BEFORE rendering the figures.
#       The bundle is self-contained: any other notebook can redraw both
#       figures from it with no scanner / joblib / GP / DD training.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from codes.nature_expscan_figures import save_figure_bundle, BUNDLE_NAME

BUNDLE_DIR  = PROJECT_ROOT / "output" / "dt_controller_fit_figures"
BUNDLE_PATH = BUNDLE_DIR / BUNDLE_NAME
save_figure_bundle(BUNDLE_PATH, tag=CACHE_TAG)
print("Saved", BUNDLE_PATH)


In [ ]:
# 9.3  Render Figures 4 & 5 from the bundle.
from codes.nature_expscan_figures import replot_figure_4, replot_figure_5
FIG_OUT = PROJECT_ROOT / "output" / "nature_figures"
fig4_paths = replot_figure_4(BUNDLE_PATH, out_dir=FIG_OUT)
fig5_paths = replot_figure_5(BUNDLE_PATH, out_dir=FIG_OUT)

from IPython.display import Image, display
display(Image(filename=str(fig4_paths["png"])))
display(Image(filename=str(fig5_paths["png"])))


## 10.  Loading the bundle in another notebook

Copy the snippet below into any notebook to redraw the experimental-scans Figures 4 & 5 from `figure_45_expscan_data_bundle.npz` alone (no DT, no joblib, no sklearn).


In [ ]:
# 10.1  Self-contained replot.  Only requires numpy, pandas, matplotlib, and
#       the two helper modules:
#         codes/nature_expscan_figures.py
#         codes/nature_statement_figures.py
from pathlib import Path
import sys
PROJECT_ROOT_FOR_LOAD = Path("/Users/richardmbp14/Library/CloudStorage/GoogleDrive-yu93liu@gmail.com/"
                              "My Drive/Shared/Coding/Digital Twins - SPM")
if str(PROJECT_ROOT_FOR_LOAD) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_FOR_LOAD))
from codes.nature_expscan_figures import (
    load_figure_bundle, replot_figure_4, replot_figure_5,
)

BUNDLE = PROJECT_ROOT_FOR_LOAD / "output" / "dt_controller_fit_figures" / "figure_45_expscan_data_bundle.npz"
b = load_figure_bundle(BUNDLE)
print("bundle keys:", sorted(k for k in b.keys() if k != "oracle_PI"))
print("traces_exp:", b["traces_exp"].shape,
      "  dt_PI:", b["dt_PI"].shape,
      "  fit_idx:", b["fit_idx"].shape,
      "  test_idx:", b["test_idx"].shape)

OUT = PROJECT_ROOT_FOR_LOAD / "output" / "nature_figures"
fig4 = replot_figure_4(BUNDLE, out_dir=OUT)
fig5 = replot_figure_5(BUNDLE, out_dir=OUT)
print(fig4["png"]); print(fig5["png"])


## 11.  Inline Figure 4 & 5 plotting (editable in this notebook)

For tweaking colours, panel layout, axis limits, etc. the Figure 4 and 5 plot
functions are reproduced **inside the notebook** so they can be edited without
touching `codes/nature_expscan_figures.py`.

This section provides:

1. `PALETTE`, `panel_label`, `save_pub` — Nature-style palette and helpers.
2. `save_data_bundle(bundle_path)` — collapse the PI grid + data-driven artifacts into one `.npz` file.
3. `load_data_bundle(bundle_path)` — inverse of the saver; returns a dict.
4. `plot_figure_4(bundle, save_to=None)` — full Figure 4 source, returns the matplotlib figure.
5. `plot_figure_5(bundle, save_to=None)` — full Figure 5 source.
6. A driver cell that runs `save → load → plot` end-to-end.

To redraw in any other notebook, copy the styling cell + `load_data_bundle` +
`plot_figure_4` + `plot_figure_5` cells, then call them with the bundle path.


In [ ]:
# 11.1  Styling helpers (palette + panel_label + save_pub).
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import json as _json
from pathlib import Path as _Path
from matplotlib import gridspec
from matplotlib.lines import Line2D

PALETTE = {
    'ink':    '#1f2933', 'muted':  '#667085',
    'blue':   '#2A6FBB', 'teal':   '#1B9E8A',
    'orange': '#D97706', 'red':    '#C2410C',
    'purple': '#7C3AED', 'grey':   '#9CA3AF', 'light': '#F3F4F6',
}

def setup_style():
    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 7.2, 'axes.labelsize': 7.4, 'axes.titlesize': 7.8,
        'xtick.labelsize': 6.6, 'ytick.labelsize': 6.6, 'legend.fontsize': 6.6,
        'axes.linewidth': 0.65,
        'axes.spines.top': False, 'axes.spines.right': False,
        'xtick.major.width': 0.55, 'ytick.major.width': 0.55,
        'xtick.major.size': 2.6, 'ytick.major.size': 2.6,
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    })

def panel_label(ax, label, x=-0.14, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, ha='left', va='top',
            fontweight='bold', fontsize=9.4, color='black')

def save_pub(fig, out_dir, stem):
    out_dir = _Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    paths = {fmt: out_dir / f'{stem}.{fmt}' for fmt in ('png', 'pdf', 'tiff')}
    fig.savefig(paths['png'], bbox_inches='tight', dpi=600, facecolor='white')
    fig.savefig(paths['pdf'], bbox_inches='tight', facecolor='white')
    fig.savefig(paths['tiff'], bbox_inches='tight', dpi=600, facecolor='white')
    return {k: str(v) for k, v in paths.items()}

def _center(arr, trim=10):
    arr = np.asarray(arr, float).copy()
    if arr.ndim == 1:
        return arr - np.nanmean(arr[trim:-trim])
    for i in range(arr.shape[0]):
        arr[i] = arr[i] - np.nanmean(arr[i, trim:-trim])
    return arr


In [ ]:
# 11.2  save_data_bundle — collapse Figure-4/5 inputs into one .npz file.
def save_data_bundle(bundle_path,
                     predictions_npz='/tmp/expscan_predictions.npz',
                     pi_grid_joblib=None,
                     summary_json_dir=None):
    """Write a self-contained .npz with every array Figures 4 & 5 consume.

    Parameters
    ----------
    bundle_path : path
        Output .npz path.
    predictions_npz : path
        Output of `codes/expscan_build_predictions.py`.
    pi_grid_joblib : path or None
        The cached PI grid joblib (default uses CACHE_TAG to locate it under
        `calibration_cache/dt_controller_fit/`).
    summary_json_dir : path or None
        If set, the Figure 5 quality summary JSON will be written here.
    """
    from joblib import load as _jload
    bundle_path = _Path(bundle_path)
    bundle_path.parent.mkdir(parents=True, exist_ok=True)

    preds = np.load(predictions_npz, allow_pickle=False)

    # Locate the PI grid joblib (carries the GP fit with train_local + res_grid)
    if pi_grid_joblib is None:
        pi_grid_joblib = (PROJECT_ROOT / 'calibration_cache' / 'dt_controller_fit'
                          / f'physics_guided_PI_grid_{CACHE_TAG}.joblib')
    pi_pack = _jload(str(pi_grid_joblib))
    gp_fit  = pi_pack['gp_fit']

    # Build cond_to_idx lookup using the predictions arrays
    si = preds['si']; di = preds['di']; spi = preds['spi']; gi = preds['gi']
    cond_to_idx = {
        (int(s), int(d), int(p), int(g)): k
        for k, (s, d, p, g) in enumerate(zip(si, di, spi, gi))
    }

    # GP local fits → oracle (P, I, log10_P, log10_I, loss) per condition idx
    gp_train = gp_fit['train_local']
    records = []
    for r in gp_train:
        key = tuple(int(v) for v in r['cond'])
        if key in cond_to_idx:
            records.append({
                'cond_idx': cond_to_idx[key],
                'P':       float(r['P']),
                'I':       float(r['I']),
                'log10_P': float(r['log10_P']),
                'log10_I': float(r['log10_I']),
                'loss':    float(r['loss']),
            })
    oracle_cond   = np.array([r['cond_idx'] for r in records], int)
    oracle_P      = np.array([r['P']        for r in records], float)
    oracle_I      = np.array([r['I']        for r in records], float)
    oracle_log10P = np.array([r['log10_P']  for r in records], float)
    oracle_log10I = np.array([r['log10_I']  for r in records], float)
    oracle_loss   = np.array([r['loss']     for r in records], float)

    # PI loss surface: pull res_grid from the lowest-loss GP-train cond that is in our list
    train_in = sorted([(i, r) for i, r in enumerate(gp_train)
                       if tuple(int(v) for v in r['cond']) in cond_to_idx],
                      key=lambda kr: float(kr[1]['loss']))
    P_grid = np.logspace(-2, 2, 13); I_grid = np.logspace(-2, 2, 13)
    L = None; cond_for_surf = None
    for _, r in train_in[:5]:
        res_grid = r.get('res_grid')
        if res_grid is None or not hasattr(res_grid, 'get'): continue
        Pg = res_grid.get('P_grid'); Ig = res_grid.get('I_grid'); Lg = res_grid.get('loss')
        if (Pg is not None and Ig is not None and Lg is not None
                and np.ndim(np.asarray(Lg)) == 2 and np.all(np.isfinite(Lg))):
            P_grid = np.asarray(Pg, float); I_grid = np.asarray(Ig, float)
            L = np.asarray(Lg, float)
            cond_for_surf = cond_to_idx[tuple(int(v) for v in r['cond'])]
            break
    if L is None and train_in:
        _, r0 = train_in[0]
        plog = np.log10(P_grid); ilog = np.log10(I_grid)
        L = (plog[:, None] - float(r0['log10_P']))**2 \
            + (ilog[None, :] - float(r0['log10_I']))**2 \
            + float(r0['loss'])
        cond_for_surf = cond_to_idx[tuple(int(v) for v in r0['cond'])]
    if cond_for_surf is None:
        cond_for_surf = int(oracle_cond[0]) if len(oracle_cond) else 0

    # h_truth — robust median of clean train traces (matches the build script).
    fit_idx_arr = preds['fit_idx']
    line_noise = np.array([np.nanstd(np.diff(preds['traces_exp'][i, 0]))
                           for i in fit_idx_arr])
    clean_pick = fit_idx_arr[np.argsort(line_noise)[:8]]
    cand = []
    for i in fit_idx_arr[:200]:
        for d in (0, 1):
            ln = preds['traces_exp'][i, d]
            if not np.all(np.isfinite(ln)): continue
            x = np.arange(ln.size, dtype=float)
            s, b = np.polyfit(x, ln, 1)
            cand.append(ln - (s*x + b))
    h_truth = np.nanmedian(np.array(cand), axis=0) if cand else np.zeros(preds['traces_exp'].shape[-1])

    np.savez(bundle_path,
        traces_exp=preds['traces_exp'],
        h_truth=h_truth,
        dt_PI=preds['dt_PI'], dd=preds['dd'], hyb=preds['hyb'],
        drive=preds['drive'], setpoint=preds['setpoint'],
        igain=preds['igain'], scan_speed=preds['scan_speed'],
        si=preds['si'], di=preds['di'], spi=preds['spi'], gi=preds['gi'],
        P_dt=preds['P_dt'], I_dt=preds['I_dt'],
        P_all=preds['P_dt'], I_all=preds['I_dt'],
        oracle_cond=oracle_cond,
        oracle_log10P=oracle_log10P, oracle_log10I=oracle_log10I,
        oracle_P=oracle_P, oracle_I=oracle_I, oracle_loss=oracle_loss,
        fit_idx=preds['fit_idx'], test_idx=preds['test_idx'],
        gp_train_idx=preds['gp_train_idx'],
        clean_pick=clean_pick,
        loss_cond=cond_for_surf, loss_P_grid=P_grid, loss_I_grid=I_grid, loss_L=L,
        q_exp=preds['q_exp'], q_PI=preds['q_PI'], q_dd=preds['q_dd'], q_hyb=preds['q_hyb'],
        rmse_PI=preds['rmse_PI'], rmse_dd=preds['rmse_dd'], rmse_hyb=preds['rmse_hyb'],
    )
    print(f'Saved {bundle_path}')
    return bundle_path


In [ ]:
# 11.3  load_data_bundle — inverse of save_data_bundle.
def load_data_bundle(bundle_path):
    """Read a bundle .npz and return a dict ready for the plot functions.

    The returned dict carries every array verbatim from the .npz, plus a
    convenience `oracle_PI` map: cond_idx -> {P, I, log10_P, log10_I, loss}.
    """
    z = np.load(bundle_path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    oracle = {}
    for i, c in enumerate(out['oracle_cond']):
        oracle[int(c)] = {
            'P':       float(out['oracle_P'][i]),
            'I':       float(out['oracle_I'][i]),
            'log10_P': float(out['oracle_log10P'][i]),
            'log10_I': float(out['oracle_log10I'][i]),
            'loss':    float(out['oracle_loss'][i]),
        }
    out['oracle_PI'] = oracle
    return out


In [ ]:
# 11.4  plot_figure_4 — fully self-contained Figure 4 for the expscan dataset.
def plot_figure_4(bundle, save_to=None, stem='figure_4_expscan_PI_fit_nature'):
    """Render Figure 4 from a bundle dict.

    Layout (3x3):
      row 1: (a) synthetic h_truth (median of cleanest exp traces) vs traces
             (b) PI loss surface for the cleanest GP-train condition
             (c) fitted log10(P,I) vs GP-predicted log10(P,I)
      row 2: (d) optimal held-out condition overlay
             (e) sub-optimal (P85) held-out condition overlay
             (f) DT-vs-exp scan-quality scatter
      row 3: (g, h, i) best / median / P85 held-out trace overlays
    """
    setup_style()
    traces_exp = bundle['traces_exp']
    h_truth = bundle['h_truth']
    fit_idx = bundle['fit_idx']; test_idx = bundle['test_idx']
    dt_PI = bundle['dt_PI']
    drive = bundle['drive']; sp = bundle['setpoint']; ig = bundle['igain']
    P_all = bundle['P_all']; I_all = bundle['I_all']
    clean_pick = bundle['clean_pick']
    q_exp = bundle['q_exp']; q_PI = bundle['q_PI']
    oracle = bundle['oracle_PI']

    fig = plt.figure(figsize=(7.2, 8.2), constrained_layout=False)
    gs = gridspec.GridSpec(3, 3, figure=fig,
                            height_ratios=[1.0, 1.05, 1.05],
                            width_ratios=[1.05, 1.0, 1.1])
    fig.subplots_adjust(left=0.085, right=0.985, bottom=0.085, top=0.88,
                        wspace=0.55, hspace=0.80)
    fig.suptitle('Calibrating the PI loop of the digital twin against experimental scan lines',
                 x=0.06, y=0.96, ha='left', fontsize=9.7, fontweight='bold')

    # (a) h_truth vs clean experimental traces
    ax = fig.add_subplot(gs[0, 0]); panel_label(ax, 'a')
    for k, ci in enumerate(clean_pick[:6]):
        ax.plot(_center(traces_exp[int(ci), 0]), lw=0.7,
                color=PALETTE['grey'], alpha=0.7,
                label='clean experimental traces' if k == 0 else None)
    ax.plot(_center(h_truth), color=PALETTE['blue'], lw=1.7,
            label=r'synthetic $h_{\mathrm{truth}}$')
    ax.set_xlabel('pixel'); ax.set_ylabel('height (nm)')
    ax.set_title('Synthetic profile mirrors\nthe topo geometry')
    ax.legend(frameon=False, loc='lower center', fontsize=6.6)

    # (b) PI loss surface for the cleanest GP-train condition
    ax = fig.add_subplot(gs[0, 1]); panel_label(ax, 'b')
    P_grid = bundle['loss_P_grid']; I_grid = bundle['loss_I_grid']; L = bundle['loss_L']
    cond_for_surf = int(bundle['loss_cond'])
    im = ax.pcolormesh(np.log10(P_grid), np.log10(I_grid),
                       np.log10(np.maximum(L, 1e-12)).T,
                       shading='auto', cmap='magma_r')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.025)
    cb.set_label(r'$\log_{10}$ loss')
    if cond_for_surf in oracle:
        p_star = oracle[cond_for_surf]['log10_P']
        i_star = oracle[cond_for_surf]['log10_I']
        ax.plot(p_star, i_star, marker='*', color=PALETTE['teal'], markersize=10,
                markeredgecolor='white', markeredgewidth=0.8,
                label=f"fitted (P*,I*) = ({oracle[cond_for_surf]['P']:.2f},{oracle[cond_for_surf]['I']:.2f})")
        ax.legend(frameon=False, fontsize=6.2, loc='lower right')
    ax.set_xlabel(r'$\log_{10} P$'); ax.set_ylabel(r'$\log_{10} I$')
    ax.set_title(f'PI loss surface\n(cond {cond_for_surf})')

    # (c) fitted (P,I) vs GP-predicted, with R^2 + best-fit slope annotation.
    # The P-gain points lie tight to y=x while the I-gain points scatter,
    # which is the headline message — quantify it inline.
    ax = fig.add_subplot(gs[0, 2]); panel_label(ax, 'c')
    o_idx = bundle['oracle_cond']
    fit_log10P = bundle['oracle_log10P']; fit_log10I = bundle['oracle_log10I']
    pred_logP = np.log10(np.maximum(P_all[o_idx], 1e-12))
    pred_logI = np.log10(np.maximum(I_all[o_idx], 1e-12))

    def _fit_stats(x, y):
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 3: return float('nan'), float('nan'), float('nan')
        x, y = x[m], y[m]
        slope, intercept = np.polyfit(x, y, 1)
        ss_res = float(np.sum((y - (slope*x + intercept))**2))
        ss_tot = float(np.sum((y - np.mean(y))**2)) + 1e-12
        r2 = 1.0 - ss_res / ss_tot
        return float(slope), float(intercept), float(r2)

    sP, iP, r2P = _fit_stats(fit_log10P, pred_logP)
    sI, iI, r2I = _fit_stats(fit_log10I, pred_logI)

    ax.scatter(fit_log10P, pred_logP, s=14, color=PALETTE['blue'],
               edgecolor='white', lw=0.4, label=f'P  ($R^2$={r2P:.2f}, slope={sP:.2f})')
    ax.scatter(fit_log10I, pred_logI, s=14, color=PALETTE['orange'],
               marker='D', edgecolor='white', lw=0.4, label=f'I  ($R^2$={r2I:.2f}, slope={sI:.2f})')
    lo = float(min(fit_log10P.min(), fit_log10I.min(), pred_logP.min(), pred_logI.min()))
    hi = float(max(fit_log10P.max(), fit_log10I.max(), pred_logP.max(), pred_logI.max()))
    xs = np.linspace(lo, hi, 64)
    # Highlight the P linear fit (this is the headline observation).
    if np.isfinite(sP):
        ax.plot(xs, sP * xs + iP, color=PALETTE['blue'], lw=1.0, alpha=0.85,
                label='P linear fit')
    ax.plot([lo, hi], [lo, hi], color=PALETTE['muted'], lw=0.6, ls='--', label='y = x')
    ax.set_xlabel(r'fitted $\log_{10}$ (P or I)')
    ax.set_ylabel(r'GP-predicted')
    ax.set_title('GP map generalizes\nacross controls')
    ax.legend(frameon=False, fontsize=6.0, loc='upper left')
    # Anchor a one-line take-away below the legend.
    ax.text(0.02, 0.04,
            r'P: tight linear map ($R^2$' + f'={r2P:.2f}); I: weaker map',
            transform=ax.transAxes, fontsize=5.8, color=PALETTE['muted'],
            ha='left', va='bottom')

    # Row 2: optimal / sub-optimal selection from the held-out set.
    # The PI loop saturates on some test conditions producing 1000+ nm outputs;
    # pick the sub-optimal at the 85th percentile (not the absolute worst).
    q_test = q_exp[test_idx]
    order_t = np.argsort(q_test)
    cond_opt = int(test_idx[order_t[0]])
    n_t = len(order_t)
    cond_sub = int(test_idx[order_t[max(0, int(0.85 * n_t) - 1)]])
    for axslot, i, ttl, c, lbl in [
        (gs[1, 0], cond_opt, 'Optimal condition',     PALETTE['blue'], 'd'),
        (gs[1, 1], cond_sub, 'Sub-optimal condition', PALETTE['red'],  'e')]:
        ax = fig.add_subplot(axslot); panel_label(ax, lbl)
        ax.plot(_center(traces_exp[i, 0]), color=PALETTE['ink'], lw=1.0)
        ax.plot(_center(traces_exp[i, 1]), color=PALETTE['ink'], lw=0.9, ls=':', alpha=0.85)
        ax.plot(_center(dt_PI[i, 0]), color=c, lw=1.0)
        ax.plot(_center(dt_PI[i, 1]), color=c, lw=0.9, ls=':', alpha=0.85)
        ax.set_xlabel('pixel'); ax.set_ylabel('height (nm)')
        ax.set_title(f'{ttl}\nd={float(drive[i]):.1f} nm, sp={float(sp[i]):.2f}, '
                     f'Ig={float(ig[i]):.0f}')

    # (f) DT-vs-exp scan-quality scatter
    ax = fig.add_subplot(gs[1, 2]); panel_label(ax, 'f')
    ax.scatter(q_exp[fit_idx], q_PI[fit_idx], s=8, color=PALETTE['blue'],
               edgecolor='white', lw=0.2, alpha=0.65, label='train')
    ax.scatter(q_exp[test_idx], q_PI[test_idx], s=18, color=PALETTE['orange'],
               marker='D', edgecolor='white', lw=0.3, label='held-out')
    qmax = float(np.nanpercentile(np.concatenate([
        q_exp[fit_idx], q_PI[fit_idx], q_exp[test_idx], q_PI[test_idx]]), 90))
    qmax = max(qmax, 8.0)
    ax.plot([0, qmax], [0, qmax], color=PALETTE['muted'], lw=0.6, ls='--')
    ax.set_xlim(0, qmax); ax.set_ylim(0, qmax)
    ax.set_xlabel('experimental scan quality (nm)')
    ax.set_ylabel('DT-simulated scan quality (nm)')
    ax.set_title('Scan-quality recovery')
    ax.legend(frameon=False, fontsize=6.4, loc='upper left')

    # Row 3: best / median / P85 held-out trace overlays, filtered for plausibility.
    def _bounded(i, max_abs=50.0):
        v = dt_PI[i][np.isfinite(dt_PI[i])]
        return v.size and float(np.max(np.abs(v))) < max_abs
    order = np.argsort(q_exp[test_idx])
    bounded_test = [int(test_idx[k]) for k in order if _bounded(int(test_idx[k]))]
    if not bounded_test:
        bounded_test = [int(test_idx[k]) for k in order]
    picks = [bounded_test[0],
             bounded_test[len(bounded_test) // 2],
             bounded_test[max(0, int(0.85 * len(bounded_test)) - 1)]]
    titles = ['best held-out', 'median held-out', 'P85 held-out']
    for col, (i, ttl) in enumerate(zip(picks, titles)):
        ax = fig.add_subplot(gs[2, col]); panel_label(ax, 'ghi'[col])
        ax.plot(_center(traces_exp[i, 0]), color=PALETTE['ink'], lw=0.9)
        ax.plot(_center(traces_exp[i, 1]), color=PALETTE['ink'], lw=0.9, ls=':', alpha=0.8)
        ax.plot(_center(dt_PI[i, 0]), color=PALETTE['blue'], lw=0.9)
        ax.plot(_center(dt_PI[i, 1]), color=PALETTE['blue'], lw=0.9, ls=':', alpha=0.8)
        ax.set_xlabel('pixel'); ax.set_ylabel('height (nm)')
        ax.set_title(f'{ttl}\nd={float(drive[i]):.1f} nm, sp={float(sp[i]):.2f}, '
                     f'Ig={float(ig[i]):.0f}')

    leg_handles = [
        Line2D([0],[0], color=PALETTE['ink'],  lw=1.0, label='experiment (trace)'),
        Line2D([0],[0], color=PALETTE['ink'],  lw=0.9, ls=':', label='experiment (retrace)'),
        Line2D([0],[0], color=PALETTE['blue'], lw=1.0, label='DT trace (PI-fit)'),
        Line2D([0],[0], color=PALETTE['blue'], lw=0.9, ls=':', label='DT retrace (PI-fit)'),
        Line2D([0],[0], color=PALETTE['red'],  lw=1.0, label='DT trace (sub-optimal cond.)'),
    ]
    fig.legend(handles=leg_handles, loc='lower center',
               bbox_to_anchor=(0.5, 0.005), ncol=5, frameon=False,
               fontsize=6.4, handletextpad=0.5, columnspacing=1.2)

    if save_to is not None:
        paths = save_pub(fig, save_to, stem)
        print('Saved Figure 4:', paths['png'])
    return fig


In [ ]:
# 11.5  plot_figure_5 — fully self-contained Figure 5 for the expscan dataset.
def plot_figure_5(bundle, save_to=None, stem='figure_5_expscan_PI_vs_hybrid_vs_DD_nature'):
    """Render Figure 5 from a bundle dict.

    Layout (2x3):
      row 1: (a) predicted vs experimental scan-quality, three methods
             (b) quality median |error| / P75 |error| bars
             (c) per-line shape RMSE boxplot (log y)
      row 2: (d, e, f) trace/retrace overlays for one held-out condition
             where the three methods disagree most (PI saturations filtered)
    """
    setup_style()
    traces_exp = bundle['traces_exp']
    test_idx = bundle['test_idx']
    dt_PI = bundle['dt_PI']; dd = bundle['dd']; hyb = bundle['hyb']
    drive = bundle['drive']; sp = bundle['setpoint']; ig = bundle['igain']
    q_exp = bundle['q_exp']; q_PI = bundle['q_PI']; q_dd = bundle['q_dd']; q_hyb = bundle['q_hyb']
    rmse_PI = bundle['rmse_PI']; rmse_dd = bundle['rmse_dd']; rmse_hyb = bundle['rmse_hyb']

    h = test_idx
    method_color = {'PI fitting':       PALETTE['blue'],
                    'PI + data-driven': PALETTE['purple'],
                    'pure data-driven': PALETTE['orange']}
    method_q    = {'PI fitting':       q_PI[h],
                   'PI + data-driven': q_hyb[h],
                   'pure data-driven': q_dd[h]}
    method_rmse = {'PI fitting':       np.nanmean(rmse_PI[h], axis=1),
                   'PI + data-driven': np.nanmean(rmse_hyb[h], axis=1),
                   'pure data-driven': np.nanmean(rmse_dd[h], axis=1)}

    fig = plt.figure(figsize=(7.2, 6.0), constrained_layout=False)
    gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1.1, 1.0],
                            width_ratios=[1.0, 1.0, 1.1])
    fig.subplots_adjust(left=0.08, right=0.985, bottom=0.13, top=0.83,
                        wspace=0.62, hspace=0.95)
    fig.suptitle('Predicting scan quality on unfitted conditions:\nPI fit vs hybrid vs pure data-driven',
                 x=0.06, y=0.965, ha='left', fontsize=9.7, fontweight='bold')

    # (a) predicted vs experimental quality
    ax = fig.add_subplot(gs[0, 0]); panel_label(ax, 'a')
    all_q = []
    for name, qm in method_q.items():
        all_q.extend(qm.tolist())
        finite = np.isfinite(qm) & np.isfinite(q_exp[h])
        ax.scatter(q_exp[h][finite], qm[finite], s=18,
                   color=method_color[name], edgecolor='white', lw=0.3,
                   alpha=0.85, label=name)
    qmax = float(np.nanpercentile([v for v in all_q if np.isfinite(v)] + q_exp[h].tolist(), 90))
    qmax = max(qmax, 6.0)
    ax.plot([0, qmax], [0, qmax], color=PALETTE['muted'], lw=0.6, ls='--')
    ax.set_xlim(0, qmax); ax.set_ylim(0, qmax)
    ax.set_xlabel('experimental quality (nm)')
    ax.set_ylabel('predicted quality (nm)')
    ax.set_title('Held-out scan-quality\n(trace–retrace RMSE)')
    ax.legend(frameon=False, fontsize=6.3, loc='upper left')

    # (b) Median / P75 quality error bars
    ax = fig.add_subplot(gs[0, 1]); panel_label(ax, 'b')
    names = list(method_q.keys())
    qe = q_exp[h]
    medae_q = [float(np.nanmedian(np.abs(qe - method_q[n]))) for n in names]
    p75_q   = [float(np.nanpercentile(np.abs(qe - method_q[n]), 75)) for n in names]
    x = np.arange(len(names)); w = 0.36
    ax.bar(x - w/2, medae_q, width=w, color=[method_color[n] for n in names],
           alpha=0.85, label='Median |error|')
    ax.bar(x + w/2, p75_q,   width=w, color=[method_color[n] for n in names],
           alpha=0.45, label='P75 |error|')
    ax.set_xticks(x, [n.replace(' + ', '\n+ ').replace(' ', '\n', 1) for n in names])
    ax.set_ylabel('quality error (nm)')
    ax.set_title('Held-out quality error')
    ax.legend(frameon=False, fontsize=6.3, loc='upper right')

    # (c) per-line shape RMSE box
    ax = fig.add_subplot(gs[0, 2]); panel_label(ax, 'c')
    box_data = [method_rmse[n][np.isfinite(method_rmse[n])] for n in names]
    bp = ax.boxplot(box_data, widths=0.55, patch_artist=True,
                    medianprops=dict(color='black', lw=1.0),
                    flierprops=dict(marker='.', markersize=2,
                                    markerfacecolor=PALETTE['muted']))
    for patch, n in zip(bp['boxes'], names):
        patch.set_facecolor(method_color[n]); patch.set_alpha(0.6)
        patch.set_edgecolor(method_color[n]); patch.set_linewidth(0.8)
    ax.set_xticks([1, 2, 3], [n.replace(' + ', '\n+ ').replace(' ', '\n', 1) for n in names])
    ax.set_ylabel('per-line RMSE vs experiment (nm)')
    ax.set_title('Held-out scan-line\nshape error')
    ax.set_yscale('log')

    # Row 2: overlay panels.  Pick a held-out condition where ALL THREE methods
    # give physically bounded predictions (no extrapolation blow-up from the
    # data-driven branches), centered in the experimental height range.  Among
    # those we prefer a condition near the median per-method RMSE so the
    # overlay is representative rather than a tail case.
    def _interior_range(arr_row, trim=10):
        v = arr_row[trim:-trim] if arr_row.size > 2*trim+4 else arr_row
        v = v[np.isfinite(v)]
        if v.size == 0: return np.inf
        return float(np.nanmax(v) - np.nanmin(v))
    def _bounded(i, max_abs_amp=30.0):
        # Require the centered range of every method to stay within twice the
        # experimental range (so data-driven extrapolation flares are excluded).
        exp_rng = _interior_range(_center(traces_exp[i, 0]))
        if not np.isfinite(exp_rng) or exp_rng < 0.05: return False
        cap = max(exp_rng * 2.5, 4.0)
        for arr in (dt_PI, hyb, dd):
            for d in (0, 1):
                v = _center(arr[i, d])
                v = v[np.isfinite(v)]
                if v.size == 0: return False
                if float(np.nanmax(np.abs(v))) > max(cap, max_abs_amp):
                    return False
        return True

    plausible = np.array([_bounded(int(i)) for i in h], bool)
    # Among plausible conditions, score by per-line RMSE (mean across methods)
    # so we land on a representative — neither best-case nor worst-case — point.
    mean_rmse = 0.5 * (method_rmse['PI fitting']
                       + 0.5 * (method_rmse['PI + data-driven']
                                + method_rmse['pure data-driven']))
    score = mean_rmse.copy()
    score[~np.isfinite(score) | ~plausible] = np.inf
    if np.any(np.isfinite(score)):
        order = np.argsort(score)
        # Use the median of plausible scores so the figure is representative,
        # not the best case (which would hide the data-driven contribution).
        valid = order[np.isfinite(score[order])]
        k = valid[len(valid) // 2]
        cond_overlay = int(h[k])
    else:
        # Fallback to the old behaviour if nothing passes the filter.
        disagree = np.abs(q_dd[h] - q_PI[h])
        disagree[~np.isfinite(disagree)] = -1
        cond_overlay = int(h[int(np.argmax(disagree))])

    for col, (name, arr) in enumerate([('PI fitting', dt_PI),
                                        ('PI + data-driven', hyb),
                                        ('pure data-driven', dd)]):
        ax = fig.add_subplot(gs[1, col]); panel_label(ax, 'def'[col])
        ax.plot(_center(traces_exp[cond_overlay, 0]), color=PALETTE['ink'], lw=1.0)
        ax.plot(_center(traces_exp[cond_overlay, 1]), color=PALETTE['ink'], lw=0.9, ls=':', alpha=0.85)
        ax.plot(_center(arr[cond_overlay, 0]), color=method_color[name], lw=1.0)
        ax.plot(_center(arr[cond_overlay, 1]), color=method_color[name], lw=0.9, ls=':', alpha=0.85)
        ax.set_xlabel('pixel'); ax.set_ylabel('height (nm)')
        ax.set_title(f'{name}\ncond {cond_overlay}\n'
                     f'(d={float(drive[cond_overlay]):.1f} nm, sp={float(sp[cond_overlay]):.2f})')

    leg_handles = [
        Line2D([0],[0], color=PALETTE['ink'],    lw=1.0, label='experiment (trace)'),
        Line2D([0],[0], color=PALETTE['ink'],    lw=0.9, ls=':', label='experiment (retrace)'),
        Line2D([0],[0], color=PALETTE['blue'],   lw=1.0, label='PI fitting (model)'),
        Line2D([0],[0], color=PALETTE['purple'], lw=1.0, label='PI + data-driven (model)'),
        Line2D([0],[0], color=PALETTE['orange'], lw=1.0, label='pure data-driven (model)'),
    ]
    fig.legend(handles=leg_handles, loc='lower center',
               bbox_to_anchor=(0.5, 0.005), ncol=5, frameon=False,
               fontsize=6.3, handletextpad=0.5, columnspacing=1.0)
    fig.text(0.985, 0.045,
             f'N held-out = {len(h)}.   Dashed line = perfect prediction.   RMSE box: log-scale.',
             ha='right', fontsize=6.0, color=PALETTE['muted'])

    if save_to is not None:
        paths = save_pub(fig, save_to, stem)
        print('Saved Figure 5:', paths['png'])
    return fig


In [ ]:
# 11.6  Save the data bundle, load it back, and render Figures 4 & 5 inline.
BUNDLE_PATH = PROJECT_ROOT / 'output' / 'dt_controller_fit_figures' / 'figure_45_expscan_data_bundle.npz'
FIG_DIR_INLINE = PROJECT_ROOT / 'output' / 'nature_figures'

# Run the prediction-build script if /tmp/expscan_predictions.npz doesn't exist yet
import os, subprocess, sys
if not os.path.exists('/tmp/expscan_predictions.npz'):
    subprocess.check_call([sys.executable,
        str(PROJECT_ROOT / 'codes/expscan_build_predictions.py'),
        '--tag', CACHE_TAG])

save_data_bundle(BUNDLE_PATH)
bundle = load_data_bundle(BUNDLE_PATH)
print('bundle keys:', sorted(k for k in bundle.keys() if k != 'oracle_PI'))

fig4 = plot_figure_4(bundle, save_to=FIG_DIR_INLINE)
fig5 = plot_figure_5(bundle, save_to=FIG_DIR_INLINE)

from IPython.display import display
display(fig4); display(fig5)


## 12.  Loading the bundle in another notebook

Copy these four cells into any other notebook to redraw both figures with
nothing but `numpy` + `matplotlib`:

* `11.1` — styling helpers (palette, panel_label, save_pub, _center)
* `11.3` — `load_data_bundle`
* `11.4` — `plot_figure_4`
* `11.5` — `plot_figure_5`

Then call:

```python
bundle = load_data_bundle(
    'output/dt_controller_fit_figures/figure_45_expscan_data_bundle.npz')
fig4 = plot_figure_4(bundle)
fig5 = plot_figure_5(bundle)
```
